In [1]:
import torch
import numpy as np
import pandas as pd

from dice import Dice, DiceFunctions
from functions import (
    MultiheadDiscreteSANetwork,
    DiscreteSANetwork
)

In [2]:
DEVICE = 'cpu'
SEED = 89290

df = pd.read_csv('./../../simulator/recsys-simulator/experiments/dataset_for_dice/log.csv')

In [3]:
sasrec_num = "0"

actions        = torch.load('./data/actions.pt', weights_only=False)
rewards        = torch.load('./data/rewards.pt', weights_only=False)
states         = torch.load(f'./data/sasrec_{0}_states.pt', weights_only=False)
target_actions = torch.load(f'./data/sasrec_{sasrec_num}_actions.pt', weights_only=False)
action_embs    = torch.load(f'./data/sasrec_{0}_action_embs.pt', weights_only=False)

In [4]:
state_dim = states[0].shape[1]
num_items = df['movieid'].unique().shape[0]
hidden_dim = 32
action_dim = action_embs.shape[1]

q_func = DiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=2,
    action_emb=action_embs,
    seed=SEED,
    device=DEVICE
)

w_func = DiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=2,
    action_emb=action_embs,
    seed=SEED,
    device=DEVICE
)

# q_func = MultiheadDiscreteSANetwork(
#     state_dim=state_dim,
#     hidden_dim=hidden_dim,
#     action_dim=num_items,
#     num_layers=2,
#     seed=SEED,
# )

# w_func = MultiheadDiscreteSANetwork(
#     state_dim=state_dim,
#     hidden_dim=hidden_dim,
#     action_dim=num_items,
#     num_layers=2,
#     seed=SEED,
# )

dice = Dice(
    q_function=q_func,
    w_function=w_func,
    gamma=0.99,
    q_lr=0.0001,
    w_lr=0.0001,
    lambda_lr=0.0001,
    f1_function=DiceFunctions.DUAL_DICE_P_3_2,
    f2_function=DiceFunctions.CHI_SQUARED,
    method_name='dual_dice',
    seed=SEED,
    device=DEVICE
)

In [5]:
dice.fit(
    state=states,
    action=actions,
    reward=rewards,
    target_action=target_actions,
    num_steps=50000,
    batch_size=8192,
    eval_iter=100,
    num_workers=4,
    result_folder=f'test',
    silent=False
)

  0%|          | 0/50000 [00:00<?, ?it/s]/Users/anthony/anaconda3/envs/dice/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
loss: -0.7140; value: 0.5149: 100%|██████████| 50000/50000 [14:53<00:00, 55.96it/s]


In [6]:
np.array(dice._experiment_data['value_per_step'])[-300:].mean()

np.float64(0.43412453057421707)

In [7]:
step_reward = dice.predict_per_step_reward(
    state=states, action=actions, reward=rewards
)

traj_reward = dice.predict_per_traj_reward(
    state=states, action=actions, reward=rewards
)

print(f"per step reward: {step_reward:.7f}")
print(f"per trajectory reward: {traj_reward:.7f}")

per step reward: 0.5149251
per trajectory reward: 77.2387663


In [8]:
w = dice.predict_weights(np.concatenate(states), np.concatenate(actions))

w.min(),w.mean(), w.max()

(np.float32(-3.9787033), np.float32(0.630328), np.float32(5.900458))